# Step 1 — Load the Data
We have 6 CSV files. We load each one into a separate table (called a DataFrame).

In [5]:
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# all CSV files live in the data folder one level up
DATA = Path('../data')

orders      = pd.read_csv(DATA / 'olist_orders_dataset.csv')
customers   = pd.read_csv(DATA / 'olist_customers_dataset.csv')
products    = pd.read_csv(DATA / 'olist_products_dataset.csv')
items       = pd.read_csv(DATA / 'olist_order_items_dataset.csv')
payments    = pd.read_csv(DATA / 'olist_order_payments_dataset.csv')
translation = pd.read_csv(DATA / 'product_category_name_translation.csv')

# print how many rows and columns each table has
for name, df in [('orders', orders), ('customers', customers),
                 ('products', products), ('items', items), ('payments', payments)]:
    rows, cols = df.shape
    print(f'{name}: {rows:,} rows, {cols} columns')

orders: 99,441 rows, 8 columns
customers: 99,441 rows, 5 columns
products: 32,951 rows, 9 columns
items: 112,650 rows, 7 columns
payments: 103,886 rows, 5 columns


# Step 2 — Check for Missing Values
Before cleaning, we want to know which columns have missing (empty) values.

In [6]:
# loop through each table and print columns that have missing values
for name, df in [('orders', orders), ('customers', customers),
                 ('products', products), ('items', items)]:
    missing = df.isnull().sum()          # count nulls per column
    missing = missing[missing > 0]       # keep only columns that actually have nulls
    if missing.empty:
        print(f'{name}: no missing values')
    else:
        print(f'\n{name}:')
        print(missing)


orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
customers: no missing values

products:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
items: no missing values


# Step 3 — Clean the Data
We only want **delivered** orders, and we need date columns to be proper dates (not just text).
We also translate product category names from Portuguese to English.

In [7]:
# --- clean orders ---

# keep only rows where the order was actually delivered
orders = orders[orders['order_status'] == 'delivered'].copy()

# convert text dates into real datetime objects so we can do math on them
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# drop rows where the purchase date is missing (we can't do time analysis without it)
orders = orders.dropna(subset=['order_purchase_timestamp'])

print(f'Delivered orders after cleaning: {len(orders):,}')

# --- translate product categories ---

# merge the translation table to add an English category column
products = products.merge(translation, on='product_category_name', how='left')

# fill any products that had no translation with 'unknown'
products['category_english'] = products['product_category_name_english'].fillna('unknown')

# drop the old Portuguese columns, we only need the English one
products = products.drop(columns=['product_category_name', 'product_category_name_english'])

print('\nTop 5 product categories:')
print(products['category_english'].value_counts().head(5))

Delivered orders after cleaning: 96,478

Top 5 product categories:
category_english
bed_bath_table     3029
sports_leisure     2867
furniture_decor    2657
health_beauty      2444
housewares         2335
Name: count, dtype: int64


# Step 4 — Build the Master Table
We combine all tables into one big table that has everything we need for analysis.
We also add useful columns like year, month, revenue, and profit.

In [8]:
# --- aggregate payments ---
# one order can have multiple payment rows, so we sum them up per order
pay_agg = (
    payments
    .groupby('order_id')['payment_value']
    .sum()
    .reset_index()
    .rename(columns={'payment_value': 'order_total_payment'})
)

# --- join all tables together ---
# start with items (one row per item sold), then add order info, customer info, etc.
master = (
    items
    .merge(orders[['order_id', 'customer_id', 'order_purchase_timestamp',
                   'order_delivered_customer_date']], on='order_id', how='inner')
    .merge(customers[['customer_id', 'customer_unique_id',
                      'customer_city', 'customer_state']], on='customer_id', how='left')
    .merge(products[['product_id', 'category_english']], on='product_id', how='left')
    .merge(pay_agg, on='order_id', how='left')
)

# --- add time-based columns ---
ts = master['order_purchase_timestamp']
master['order_date'] = ts.dt.date
master['year']       = ts.dt.year
master['month']      = ts.dt.month
master['quarter']    = ts.dt.quarter
master['weekday']    = ts.dt.day_name()

# --- add revenue and profit columns ---
master['revenue'] = master['price'] + master['freight_value']  # total paid by customer
master['profit']  = master['price'] * 0.20                     # assume 20% margin

# --- remove duplicates and bad rows ---
before = len(master)
master = master.drop_duplicates(subset=['order_id', 'order_item_id', 'product_id'])
master = master[master['price'] > 0]   # remove rows with zero or negative price

print(f'Duplicate rows removed: {before - len(master)}')
print(f'Final table: {len(master):,} rows, {master.shape[1]} columns')

# --- save to CSV ---
master.to_csv(DATA / 'master.csv', index=False)
print('Saved master.csv')

Duplicate rows removed: 0
Final table: 110,197 rows, 22 columns
Saved master.csv
